# Aggregate Tier-1 Results

Coverage and public aggregate comparisons for all seven Tier-1 methods.

This executed notebook uses only public prompts, responses, and aggregate analysis. It does not embed answer keys or row-level correctness labels.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
manifest_path = root / 'tier1_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}

def read_public_csv(path):
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, keep_default_na=False)
    forbidden = {'gold', 'ground_truth', 'correct_answer', 'normalized_gold', 'is_correct'}
    exposed = forbidden.intersection(str(column).lower() for column in frame.columns)
    if exposed:
        raise ValueError(f'public analysis contains forbidden row-level columns: {sorted(exposed)}')
    return frame

print('model:', manifest.get('model', 'not prepared'))
print('reasoning effort:', manifest.get('reasoning_effort', 'not prepared'))
print('prompt version:', manifest.get('prompt_version', 'not prepared'))


model: gpt-5.6-sol
reasoning effort: high
prompt version: gpt56-tier1-v1


In [2]:
expected = {
    'normalize_then_solve': 250,
    'structured_decomposition': 250,
    'explicit_task_instructions': 200,
    'chain_of_thought': 450,
    'rereading_s2a': 200,
    'self_consistency_path_1': 450,
    'self_consistency_path_2': 450,
    'self_consistency_path_3': 450,
    'self_verification_cove_initial': 450,
    'self_verification_cove_verification': 450,
}
coverage = []
for queue, expected_rows in expected.items():
    dataset = read_public_csv(root / 'Dataset' / f'{queue}.csv')
    result = read_public_csv(root / 'Result' / f'{queue}.csv')
    answered = 0 if result.empty else int(result['model_response'].astype(str).str.strip().ne('').sum())
    coverage.append({
        'queue': queue,
        'expected_rows': expected_rows,
        'dataset_rows': len(dataset),
        'result_rows': len(result),
        'answered_rows': answered,
    })
coverage_frame = pd.DataFrame(coverage)
display(coverage_frame)
print('planned scored evaluation units:', sum((250, 250, 200, 450, 200, 450, 450)))
print('planned raw generations:', sum(expected.values()))

analysis_dir = root / 'Analysis'
if analysis_dir.exists():
    for path in sorted(analysis_dir.glob('*.csv')):
        frame = read_public_csv(path)
        print(path.name)
        if 'evaluation_id' in frame.columns and len(frame) > 100:
            print('public row-level aggregate rows:', len(frame))
            grouping = [column for column in ('condition', 'task', 'variant', 'agreement', 'status', 'answer_changed') if column in frame]
            if grouping:
                display(frame.groupby(grouping, dropna=False).size().rename('rows').reset_index())
        else:
            display(frame)
    summary_path = analysis_dir / 'summary.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        safe_keys = {
            'model', 'reasoning_effort', 'prompt_version', 'scoring_policy',
            'conditions', 'task_variant_cells', 'evaluation_units',
            'raw_model_generations', 'completed_evaluations', 'condition_totals',
            'special_metrics', 'status_counts',
        }
        display({key: value for key, value in summary.items() if key in safe_keys})
    aggregation_path = analysis_dir / 'aggregation_summary.json'
    if aggregation_path.exists():
        display(json.loads(aggregation_path.read_text(encoding='utf-8')))
else:
    print('Analysis directory is not available yet.')


,queue,expected_rows,dataset_rows,result_rows,answered_rows
0,normalize_then_solve,250,250,250,250
1,structured_decomposition,250,250,250,250
2,explicit_task_instructions,200,200,200,200
3,chain_of_thought,450,450,450,450
4,rereading_s2a,200,200,200,200
5,self_consistency_path_1,450,450,450,450
6,self_consistency_path_2,450,450,450,450
7,self_consistency_path_3,450,450,450,450
8,self_verification_cove_initial,450,450,450,450
9,self_verification_cove_verification,450,450,450,450


planned scored evaluation units: 2250
planned raw generations: 3600
accuracy_by_cell.csv


,model,condition,task,task_title,variant,correct,total,accuracy_pct,response_or_consensus_failures
0,gpt-5.6-sol,normalize_then_solve,obfus_fol,FOL,base,44,50,88.0,0
1,gpt-5.6-sol,normalize_then_solve,obfus_fol,FOL,obfuscation,33,50,66.0,0
2,gpt-5.6-sol,normalize_then_solve,obfus_blood_relation,Blood Relation,base,39,50,78.0,0
3,gpt-5.6-sol,normalize_then_solve,obfus_blood_relation,Blood Relation,obfuscation_l1,34,50,68.0,0
4,gpt-5.6-sol,normalize_then_solve,obfus_blood_relation,Blood Relation,obfuscation_l2,31,50,62.0,0
5,gpt-5.6-sol,structured_decomposition,obfus_blood_relation,Blood Relation,base,39,50,78.0,0
6,gpt-5.6-sol,structured_decomposition,obfus_blood_relation,Blood Relation,obfuscation_l1,34,50,68.0,0
7,gpt-5.6-sol,structured_decomposition,obfus_blood_relation,Blood Relation,obfuscation_l2,31,50,62.0,0
8,gpt-5.6-sol,structured_decomposition,obfus_direction_sense,Direction Sense,base,23,50,46.0,0
9,gpt-5.6-sol,structured_decomposition,obfus_direction_sense,Direction Sense,obfuscation,19,50,38.0,0


base_obfuscated_gaps.csv


,condition,task,variant,base_source_condition,base_correct,obfuscated_correct,total,base_accuracy_pct,obfuscated_accuracy_pct,base_minus_obfuscated_pp
0,normalize_then_solve,obfus_fol,obfuscation,normalize_then_solve,44,33,50,88.0,66.0,22.0
1,normalize_then_solve,obfus_blood_relation,obfuscation_l1,normalize_then_solve,39,34,50,78.0,68.0,10.0
2,normalize_then_solve,obfus_blood_relation,obfuscation_l2,normalize_then_solve,39,31,50,78.0,62.0,16.0
3,structured_decomposition,obfus_blood_relation,obfuscation_l1,structured_decomposition,39,34,50,78.0,68.0,10.0
4,structured_decomposition,obfus_blood_relation,obfuscation_l2,structured_decomposition,39,31,50,78.0,62.0,16.0
5,structured_decomposition,obfus_direction_sense,obfuscation,structured_decomposition,23,19,50,46.0,38.0,8.0
6,explicit_task_instructions,obfus_number_series,obfuscation,explicit_task_instructions,41,41,50,82.0,82.0,0.0
7,explicit_task_instructions,obfus_direction_sense,obfuscation,explicit_task_instructions,24,19,50,48.0,38.0,10.0
8,chain_of_thought,obfus_fol,obfuscation,chain_of_thought,43,34,50,86.0,68.0,18.0
9,chain_of_thought,obfus_blood_relation,obfuscation_l1,chain_of_thought,42,41,50,84.0,82.0,2.0


...

,condition,task,variant,both_correct,base_only_correct,obfuscated_only_correct,both_wrong,total,paired_retention_pct
0,normalize_then_solve,obfus_fol,obfuscation,32,12,1,5,50,72.7273
1,normalize_then_solve,obfus_blood_relation,obfuscation_l1,34,5,0,11,50,87.1795
2,normalize_then_solve,obfus_blood_relation,obfuscation_l2,31,8,0,11,50,79.4872
3,structured_decomposition,obfus_blood_relation,obfuscation_l1,34,5,0,11,50,87.1795
4,structured_decomposition,obfus_blood_relation,obfuscation_l2,31,8,0,11,50,79.4872
5,structured_decomposition,obfus_direction_sense,obfuscation,18,5,1,26,50,78.2609
6,explicit_task_instructions,obfus_number_series,obfuscation,41,0,0,9,50,100.0000
7,explicit_task_instructions,obfus_direction_sense,obfuscation,19,5,0,26,50,79.1667
8,chain_of_thought,obfus_fol,obfuscation,33,10,1,6,50,76.7442
9,chain_of_thought,obfus_blood_relation,obfuscation_l1,41,1,0,8,50,97.6190


self_consistency_aggregates.csv
public row-level aggregate rows: 450


,condition,task,variant,agreement,status,rows
0,self_consistency,obfus_blood_relation,base,majority,answered,3
1,self_consistency,obfus_blood_relation,base,unanimous,answered,47
2,self_consistency,obfus_blood_relation,obfuscation_l1,majority,answered,5
3,self_consistency,obfus_blood_relation,obfuscation_l1,none,no_consensus,2
4,self_consistency,obfus_blood_relation,obfuscation_l1,unanimous,answered,43
5,self_consistency,obfus_blood_relation,obfuscation_l2,majority,answered,6
6,self_consistency,obfus_blood_relation,obfuscation_l2,none,no_consensus,1
7,self_consistency,obfus_blood_relation,obfuscation_l2,unanimous,answered,43
8,self_consistency,obfus_direction_sense,base,majority,answered,17
9,self_consistency,obfus_direction_sense,base,unanimous,answered,33


{'model': 'gpt-5.6-sol',
 'reasoning_effort': 'high',
 'conditions': 7,
 'task_variant_cells': 45,
 'evaluation_units': 2250,
 'raw_model_generations': 3600,
 'completed_evaluations': 2250,
 'status_counts': {'answered': 2243, 'no_consensus': 7},
 'scoring_policy': 'Parent-package task normalizers with digit/word Blood Relation count equivalence; Direction distance tolerance is 0.05.',
 'condition_totals': [{'condition': 'normalize_then_solve',
   'scope': 'all applicable variants',
   'correct': 181,
   'total': 250,
   'accuracy_pct': 72.4},
  {'condition': 'normalize_then_solve',
   'scope': 'obfuscated variants',
   'correct': 98,
   'total': 150,
   'accuracy_pct': 65.3333},
  {'condition': 'structured_decomposition',
   'scope': 'all applicable variants',
   'correct': 146,
   'total': 250,
   'accuracy_pct': 58.4},
  {'condition': 'structured_decomposition',
   'scope': 'obfuscated variants',
   'correct': 84,
   'total': 150,
   'accuracy_pct': 56.0},
  {'condition': 'explicit_

{'self_consistency_evaluations': 450,
 'self_consistency_raw_paths': 1350,
 'self_consistency_agreement': {'unanimous': 382, 'majority': 61, 'none': 7},
 'cove_evaluations': 450,
 'cove_raw_passes': 900,
 'cove_answers_changed': 6}